# Notebook 03 — Rivers & Hydrological Factors

## Objective

This notebook prepares the river- and drainage-related flood-conditioning
factors for the Gomti River–influenced Lucknow study area.

The notebook will use the verified Gomti River network from Notebook 01
and the DEM-derived hydrological information produced in Notebook 02.

The main objectives are:

1. Verify the input river and hydrological datasets.
2. Calculate distance to the Gomti River.
3. Inspect the existing flow-accumulation raster.
4. Determine whether a drainage network can be derived from flow accumulation.
5. Calculate drainage density if the derived drainage network is
   scientifically appropriate.
6. Validate the resulting factors before passing them to later notebooks.

No previously completed hydrological calculation will be repeated.

## Why River and Hydrological Factors Matter

Flood susceptibility is influenced not only by terrain elevation and slope,
but also by how water is distributed and concentrated across the landscape.

River-related and hydrological factors provide information about the
potential influence of surface-water flow and drainage conditions.

The main factors considered in this notebook are:

- Distance to the Gomti River
- Flow accumulation
- Drainage density, if justified

### Distance to River

Locations closer to the river may have greater potential exposure to
river-related flooding.

### Flow Accumulation

Flow accumulation represents the number of upstream cells contributing
flow toward a location. Areas with high flow accumulation indicate
preferential pathways where surface runoff converges.

This factor has already been calculated in Notebook 02 and will be reused.

### Drainage Density

Drainage density describes the concentration of drainage channels within
an area. Higher drainage density can indicate a more developed drainage
network and different runoff characteristics.

Drainage density will only be included if it can be derived reliably from
the available data.

## Input Data from Previous Notebooks

Notebook 03 reuses the outputs already generated in Notebooks 01 and 02.

### From Notebook 01

- Verified Gomti River network
- Lucknow + 20 km study area
- Common projected CRS: EPSG:32644

The Gomti River was identified from HydroRIVERS using:

`MAIN_RIV = 41067217`

### From Notebook 02

- Flow accumulation raster
- Flow direction raster
- Hydrologically conditioned DEM
- Study-area DEM

These datasets are already available locally in:

`data/processed/`

Large raster files are intentionally not stored in the GitHub repository.

In [ ]:
# Core libraries
from pathlib import Path

# Vector data
import geopandas as gpd

# Raster data
import rasterio
import numpy as np

# Plotting
import matplotlib.pyplot as plt

## 1. Input Data Validation

Before calculating any new factor, the input datasets must be inspected.

We will verify:

- File existence
- CRS
- Raster dimensions
- Pixel resolution
- Spatial extent
- NoData values
- Minimum and maximum values
- Geometry type and feature count for the river network

This step is necessary because downstream calculations are only reliable
when the input datasets are spatially compatible.

In [ ]:
# Project root
PROJECT_ROOT = Path.cwd().parent

# Processed data directory
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Notebook-specific output directory
NOTEBOOK_03_DIR = PROCESSED_DIR / "notebook_03"

# Create Notebook 03 output directory if it does not exist
NOTEBOOK_03_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:")
print(PROJECT_ROOT)

print("\nProcessed data directory:")
print(PROCESSED_DIR)

print("\nNotebook 03 output directory:")
print(NOTEBOOK_03_DIR)

In [ ]:
print("Contents of data/processed:\n")

for item in sorted(PROCESSED_DIR.rglob("*")):
    if item.is_file():
        print(item.relative_to(PROCESSED_DIR))

## 2. Gomti River Network Validation

The Gomti River network identified in Notebook 01 will be used as the
reference river layer for the river-distance analysis.

We will verify:

- Number of river reaches
- Geometry type
- CRS
- Spatial extent
- Whether the river network overlaps the defined study area

The purpose of this check is to ensure that the correct river network is
being used before calculating distance from the river.

In [ ]:
# Path to the verified Gomti River network from Notebook 01
RIVER_FILE = (
    PROCESSED_DIR
    / "notebook_01"
    / "Gomti_River_Network_Lucknow.gpkg"
)

# Check that the file exists
print("River file:")
print(RIVER_FILE)

print("\nFile exists:", RIVER_FILE.exists())

In [ ]:
# Load the Gomti River network
river = gpd.read_file(RIVER_FILE)

print("River layer loaded successfully.")
print("Number of features:", len(river))
print("CRS:", river.crs)

print("\nGeometry types:")
print(river.geom_type.value_counts())

In [ ]:
print("River layer columns:")
print(river.columns.tolist())

In [ ]:
# Verify MAIN_RIV
if "MAIN_RIV" in river.columns:

    print("Unique MAIN_RIV values:")
    print(river["MAIN_RIV"].unique())

    print("\nNumber of unique MAIN_RIV values:")
    print(river["MAIN_RIV"].nunique())

else:
    print("MAIN_RIV field was not found.")

In [ ]:
print("Gomti River spatial extent:")
print(river.total_bounds)

## 3. Flow Accumulation Validation

Flow accumulation was already calculated in Notebook 02.

It will not be recalculated here.

Instead, we will inspect the existing raster to confirm that:

- The raster can be opened correctly.
- Its CRS is appropriate.
- Its resolution is known.
- Its spatial extent is compatible with the study area.
- NoData values are handled correctly.
- The value distribution is reasonable.
- High-accumulation areas form a coherent drainage pattern.

This check determines whether the existing flow-accumulation product is
suitable for subsequent drainage-network analysis.

In [ ]:
# Path to the existing flow-accumulation raster from Notebook 02
FLOW_ACC_FILE = (
    PROCESSED_DIR
    / "notebook_02"
    / "Flow_Accumulation_Lucknow_conditioned.tif"
)

print("Flow accumulation file:")
print(FLOW_ACC_FILE)

print("\nFile exists:", FLOW_ACC_FILE.exists())

In [ ]:
with rasterio.open(FLOW_ACC_FILE) as src:

    print("CRS:", src.crs)
    print("Width:", src.width)
    print("Height:", src.height)
    print("Resolution:", src.res)
    print("Bounds:", src.bounds)
    print("NoData:", src.nodata)
    print("Data type:", src.dtypes[0])
    print("Number of bands:", src.count)

In [ ]:
with rasterio.open(FLOW_ACC_FILE) as src:

    flow_acc = src.read(1)
    nodata = src.nodata

# Remove NoData values
if nodata is not None:
    valid_flow_acc = flow_acc[flow_acc != nodata]
else:
    valid_flow_acc = flow_acc[np.isfinite(flow_acc)]

print("Valid cells:", valid_flow_acc.size)
print("Minimum:", np.min(valid_flow_acc))
print("Maximum:", np.max(valid_flow_acc))
print("Mean:", np.mean(valid_flow_acc))
print("Median:", np.median(valid_flow_acc))

## 4. Spatial Reference Compatibility

The Gomti River network and the flow-accumulation raster must use a
compatible projected coordinate reference system before spatial analysis.

The project uses EPSG:32644 (UTM Zone 44N), which provides metre-based
coordinates suitable for distance calculations around Lucknow.

We will verify that the river and raster use the same CRS.

In [ ]:
# River CRS
river_crs = river.crs

# Flow accumulation CRS
with rasterio.open(FLOW_ACC_FILE) as src:
    flow_acc_crs = src.crs

print("River CRS:")
print(river_crs)

print("\nFlow accumulation CRS:")
print(flow_acc_crs)

print("\nCRS match:", river_crs == flow_acc_crs)

## 5. Spatial Coverage Check

The river network and flow-accumulation raster should cover the same
analysis region.

We therefore compare their spatial extents before performing distance
and drainage analysis.

This prevents errors caused by using layers covering different geographic
areas.

In [ ]:
with rasterio.open(FLOW_ACC_FILE) as src:
    raster_bounds = src.bounds

print("Gomti River bounds:")
print(river.total_bounds)

print("\nFlow accumulation bounds:")
print(raster_bounds)

## 6. Visual Validation of Flow Accumulation

Numerical statistics alone are not sufficient to validate a hydrological
raster.

Flow accumulation should produce a coherent drainage structure, with
higher values following concentrated flow paths.

A logarithmic transformation is used only for visualization because flow
accumulation values can span several orders of magnitude.

The original raster values are not modified.

In [ ]:
with rasterio.open(FLOW_ACC_FILE) as src:

    flow_acc_plot = src.read(1)
    nodata = src.nodata

# Mask NoData
if nodata is not None:
    flow_acc_plot = np.ma.masked_equal(flow_acc_plot, nodata)

# Log transformation ONLY for visualization
flow_acc_log = np.ma.log10(flow_acc_plot)

fig, ax = plt.subplots(figsize=(10, 8))

ax.imshow(
    flow_acc_log,
    cmap="viridis",
    extent=(
        raster_bounds.left,
        raster_bounds.right,
        raster_bounds.bottom,
        raster_bounds.top,
    ),
    origin="upper",
)

ax.set_title("Flow Accumulation — Visual Validation")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")

plt.tight_layout()
plt.show()

## 7. Gomti River and Flow Accumulation Comparison

The DEM-derived flow-accumulation pattern is compared visually with the
verified Gomti River network.

The purpose is qualitative spatial validation.

We are not treating this comparison as a formal accuracy assessment of
the DEM or river dataset.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

with rasterio.open(FLOW_ACC_FILE) as src:

    flow_acc_plot = src.read(1)
    nodata = src.nodata

    if nodata is not None:
        flow_acc_plot = np.ma.masked_equal(
            flow_acc_plot,
            nodata
        )

    flow_acc_log = np.ma.log10(flow_acc_plot)

    ax.imshow(
        flow_acc_log,
        cmap="viridis",
        extent=(
            raster_bounds.left,
            raster_bounds.right,
            raster_bounds.bottom,
            raster_bounds.top,
        ),
        origin="upper",
    )

# Plot Gomti River
river.plot(
    ax=ax,
    facecolor="none",
    edgecolor="red",
    linewidth=1
)

ax.set_title("Flow Accumulation and Gomti River")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")

plt.tight_layout()
plt.show()

## 8. Save Diagnostic Outputs

Diagnostic figures generated during Notebook 03 are saved in the
Notebook 03 processed-data directory.

Large raster outputs are retained locally and are not committed to GitHub.

In [ ]:
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures" / "notebook_03"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

output_figure = FIGURES_DIR / "flow_accumulation_and_gomti_river.png"

fig, ax = plt.subplots(figsize=(10, 8))

with rasterio.open(FLOW_ACC_FILE) as src:

    flow_acc_plot = src.read(1)
    nodata = src.nodata
    transform = src.transform

    if nodata is not None:
        flow_acc_plot = np.ma.masked_equal(
            flow_acc_plot,
            nodata
        )

    flow_acc_log = np.ma.log10(flow_acc_plot)

    ax.imshow(
        flow_acc_log,
        transform=ax.transData,
        extent=(
            raster_bounds.left,
            raster_bounds.right,
            raster_bounds.bottom,
            raster_bounds.top,
        ),
        origin="upper",
        cmap="viridis",
    )

river.plot(
    ax=ax,
    facecolor="none",
    edgecolor="red",
    linewidth=1
)

ax.set_title("Flow Accumulation and Gomti River")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")

plt.tight_layout()

fig.savefig(
    output_figure,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved:", output_figure)

## 9. River and Analysis Raster Spatial Coverage

Before calculating distance to the Gomti River, the spatial coverage of
the river network and analysis raster must be examined.

The river network obtained from Notebook 01 may contain reaches extending
beyond the spatial extent of the DEM-derived analysis raster.

For the distance-to-river factor, the river geometry must be spatially
consistent with the area over which the distance raster will be calculated.

We therefore inspect the spatial relationship between the river network
and the flow-accumulation raster before performing the distance calculation.

This step prevents river segments outside the intended analysis domain from
being unintentionally included in the distance calculation.

In [ ]:
# Get the flow-accumulation raster bounds
with rasterio.open(FLOW_ACC_FILE) as src:
    raster_bounds = src.bounds

# Convert raster bounds into a GeoDataFrame
raster_extent = gpd.GeoDataFrame(
    {
        "geometry": [
            gpd.GeoSeries.from_wkt(
                [
                    f"POLYGON (("
                    f"{raster_bounds.left} {raster_bounds.bottom}, "
                    f"{raster_bounds.right} {raster_bounds.bottom}, "
                    f"{raster_bounds.right} {raster_bounds.top}, "
                    f"{raster_bounds.left} {raster_bounds.top}, "
                    f"{raster_bounds.left} {raster_bounds.bottom}"
                    f"))"
                ]
            ).iloc[0]
        ]
    },
    crs=flow_acc_crs
)

print("Flow accumulation bounds:")
print(raster_bounds)

print("\nRiver bounds:")
print(river.total_bounds)

print("\nRiver CRS:")
print(river.crs)

print("\nRaster CRS:")
print(flow_acc_crs)

In [ ]:
from shapely.geometry import box

# Get raster bounds
with rasterio.open(FLOW_ACC_FILE) as src:
    raster_bounds = src.bounds

# Create raster extent polygon
raster_extent_geom = box(
    raster_bounds.left,
    raster_bounds.bottom,
    raster_bounds.right,
    raster_bounds.top
)

raster_extent = gpd.GeoDataFrame(
    {"geometry": [raster_extent_geom]},
    crs=flow_acc_crs
)

print("Flow accumulation bounds:")
print(raster_bounds)

print("\nRiver bounds:")
print(river.total_bounds)

In [ ]:
# Ensure the river uses the raster CRS
river_for_check = river.to_crs(flow_acc_crs)

# Clip river geometries to the raster extent
river_inside_raster = gpd.clip(
    river_for_check,
    raster_extent
)

print("Original river reaches:", len(river_for_check))
print("River features intersecting raster:", len(river_inside_raster))

print("\nOriginal total river length (m):")
print(river_for_check.length.sum())

print("\nRiver length inside raster (m):")
print(river_inside_raster.length.sum())

## 10. Distance to the Gomti River

Distance to river represents the shortest horizontal distance from each
location in the analysis area to the Gomti River network.

The calculation is based on the spatial geometry of the river network and
does not require elevation values from outside the analysis raster.

The verified 160-reach Gomti River network from Notebook 01 will be used
as the reference river geometry.

The distance calculation will be performed in EPSG:32644 (UTM Zone 44N),
where coordinates are expressed in metres. Therefore, the resulting
distance raster will have distance values in metres.

Lower distance values indicate locations closer to the Gomti River, while
higher values indicate locations farther from the river.

The output will be restricted to the existing analysis raster grid so that
the resulting factor can later be harmonized with the other flood-
conditioning factors.

In [ ]:
# Reproject the verified Gomti River network to the analysis CRS
river_distance = river.to_crs(flow_acc_crs)

print("River CRS:", river_distance.crs)
print("Number of river reaches:", len(river_distance))
print("Geometry types:")
print(river_distance.geom_type.value_counts())

In [ ]:
# Read the spatial properties of the existing analysis raster

with rasterio.open(FLOW_ACC_FILE) as src:

    analysis_profile = src.profile.copy()

    analysis_crs = src.crs
    analysis_transform = src.transform
    analysis_width = src.width
    analysis_height = src.height
    analysis_bounds = src.bounds
    analysis_resolution = src.res

print("Analysis CRS:", analysis_crs)
print("Raster dimensions:", analysis_width, "×", analysis_height)
print("Resolution:", analysis_resolution)
print("Bounds:", analysis_bounds)

In [ ]:
from rasterio.features import rasterize

# Create an empty raster grid matching the flow-accumulation raster
river_raster = rasterize(
    [(geom, 1) for geom in river_distance.geometry],
    out_shape=(analysis_height, analysis_width),
    transform=analysis_transform,
    fill=0,
    dtype="uint8"
)

print("River raster shape:", river_raster.shape)
print("Unique values:", np.unique(river_raster))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

ax.imshow(
    river_raster,
    cmap="Blues"
)

ax.set_title("Rasterized Gomti River")
ax.set_xlabel("Column")
ax.set_ylabel("Row")

plt.tight_layout()
plt.show()

In [ ]:
from scipy.ndimage import distance_transform_edt

# Calculate Euclidean distance from every cell to the nearest river cell

distance_to_river_cells = distance_transform_edt(
    river_raster == 0,
    sampling=analysis_resolution
)

print("Minimum distance:", distance_to_river_cells.min(), "m")
print("Maximum distance:", distance_to_river_cells.max(), "m")
print("Mean distance:", distance_to_river_cells.mean(), "m")

In [ ]:
river_cell_distances = distance_to_river_cells[river_raster == 1]

print("Minimum distance on river cells:", river_cell_distances.min())
print("Maximum distance on river cells:", river_cell_distances.max())

## 11. Distance-to-River Raster Creation

The Euclidean distance calculated from the rasterized Gomti River is
converted into a GeoTIFF using the same spatial grid as the existing
flow-accumulation raster.

Using the existing raster as the spatial template ensures that the
distance-to-river factor has the same:

- CRS
- spatial extent
- resolution
- width and height
- pixel alignment

as the existing hydrological analysis grid.

Distance values are stored in metres.

In [ ]:
# Output path for the distance-to-river factor
DISTANCE_RIVER_FILE = (
    NOTEBOOK_03_DIR
    / "Distance_to_Gomti_River.tif"
)

# Create output profile based on the analysis raster
distance_profile = analysis_profile.copy()

distance_profile.update(
    dtype="float32",
    count=1,
    compress="lzw",
    nodata=-9999.0
)

# Write the distance raster
with rasterio.open(
    DISTANCE_RIVER_FILE,
    "w",
    **distance_profile
) as dst:

    dst.write(
        distance_to_river_cells.astype("float32"),
        1
    )

print("Distance-to-river raster saved:")
print(DISTANCE_RIVER_FILE)

## 12. Distance-to-River Validation

The newly generated distance-to-river raster is validated before being
used in subsequent flood-susceptibility analysis.

The following properties are checked:

- CRS
- raster dimensions
- resolution
- NoData value
- minimum distance
- maximum distance
- mean distance
- distance values at river cells

The spatial pattern is also visually inspected to confirm that distance
increases progressively away from the Gomti River.

In [ ]:
with rasterio.open(DISTANCE_RIVER_FILE) as src:

    distance_data = src.read(1)

    print("CRS:", src.crs)
    print("Width:", src.width)
    print("Height:", src.height)
    print("Resolution:", src.res)
    print("Bounds:", src.bounds)
    print("NoData:", src.nodata)
    print("Data type:", src.dtypes[0])

    valid_distance = distance_data[
        distance_data != src.nodata
    ]

    print("\nDistance statistics:")
    print("Minimum:", valid_distance.min(), "m")
    print("Maximum:", valid_distance.max(), "m")
    print("Mean:", valid_distance.mean(), "m")

In [ ]:
with rasterio.open(FLOW_ACC_FILE) as flow_src:
    with rasterio.open(DISTANCE_RIVER_FILE) as distance_src:

        print("CRS identical:",
              flow_src.crs == distance_src.crs)

        print("Width identical:",
              flow_src.width == distance_src.width)

        print("Height identical:",
              flow_src.height == distance_src.height)

        print("Resolution identical:",
              flow_src.res == distance_src.res)

        print("Transform identical:",
              flow_src.transform == distance_src.transform)

        print("Bounds identical:",
              flow_src.bounds == distance_src.bounds)

## 13. Visual Validation of Distance to the Gomti River

The distance-to-river raster is visualized to verify its spatial pattern.

Locations immediately adjacent to the Gomti River should have the lowest
distance values, while distances should generally increase away from the
river.

The visualization is a spatial sanity check and does not represent the
final flood-susceptibility classification.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

with rasterio.open(DISTANCE_RIVER_FILE) as src:

    distance_data = src.read(1)
    nodata = src.nodata

    if nodata is not None:
        distance_data = np.ma.masked_equal(
            distance_data,
            nodata
        )

    ax.imshow(
        distance_data,
        extent=(
            analysis_bounds.left,
            analysis_bounds.right,
            analysis_bounds.bottom,
            analysis_bounds.top,
        ),
        origin="upper",
        cmap="viridis",
    )

ax.set_title("Distance to Gomti River")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

with rasterio.open(DISTANCE_RIVER_FILE) as src:

    distance_data = src.read(1)
    nodata = src.nodata

    if nodata is not None:
        distance_data = np.ma.masked_equal(
            distance_data,
            nodata
        )

    ax.imshow(
        distance_data,
        extent=(
            analysis_bounds.left,
            analysis_bounds.right,
            analysis_bounds.bottom,
            analysis_bounds.top,
        ),
        origin="upper",
        cmap="viridis",
    )

river_distance.plot(
    ax=ax,
    facecolor="none",
    edgecolor="red",
    linewidth=1
)

ax.set_title("Distance to Gomti River with River Network")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")

plt.tight_layout()

plt.savefig(FIGURES_DIR/ "Distance to Gomti River with River Network")
plt.show()

## 14. Drainage Network Extraction

### Concept

Flow accumulation represents the amount of upstream contributing area
associated with each raster cell.

Cells with sufficiently high flow accumulation can be interpreted as
preferential drainage pathways.

A drainage network can therefore be derived by applying a threshold to the
flow-accumulation raster:

Flow Accumulation
        ↓
Threshold
        ↓
Drainage Network

Cells with flow accumulation equal to or greater than the selected
threshold are classified as drainage cells.

### Threshold Selection

The threshold controls the density of the extracted drainage network.

A very low threshold may produce an unrealistically dense network,
whereas a very high threshold may omit important drainage channels.

Therefore, the threshold will not be selected arbitrarily. Multiple
candidate thresholds will first be evaluated using both contributing-area
interpretation and visual inspection of the resulting drainage patterns.

The selected threshold will be retained as a methodological parameter and
documented for reproducibility.

The flow-accumulation raster from Notebook 02 is reused directly; it is not
recalculated in this notebook.

In [ ]:
# Load the existing flow-accumulation raster

with rasterio.open(FLOW_ACC_FILE) as src:

    flow_acc = src.read(1)
    flow_acc_profile = src.profile.copy()
    flow_acc_transform = src.transform
    flow_acc_crs = src.crs
    flow_acc_nodata = src.nodata
    flow_acc_resolution = src.res

# Mask NoData values
if flow_acc_nodata is not None:
    flow_acc_valid = np.ma.masked_equal(
        flow_acc,
        flow_acc_nodata
    )
else:
    flow_acc_valid = np.ma.masked_invalid(flow_acc)

print("Flow accumulation CRS:", flow_acc_crs)
print("Resolution:", flow_acc_resolution)
print("Minimum:", flow_acc_valid.min())
print("Maximum:", flow_acc_valid.max())
print("Mean:", flow_acc_valid.mean())

In [ ]:
# Calculate area represented by one raster cell

cell_width = abs(flow_acc_resolution[0])
cell_height = abs(flow_acc_resolution[1])

cell_area_m2 = cell_width * cell_height
cell_area_km2 = cell_area_m2 / 1_000_000

print(f"Cell width: {cell_width:.3f} m")
print(f"Cell height: {cell_height:.3f} m")
print(f"Cell area: {cell_area_m2:.3f} m²")
print(f"Cell area: {cell_area_km2:.6f} km²")

In [ ]:
# Candidate flow-accumulation thresholds

thresholds = [
    100,
    500,
    1_000,
    5_000,
    10_000,
    50_000,
    100_000
]

print("Candidate thresholds:\n")

for threshold in thresholds:

    contributing_area = (
        threshold * cell_area_km2
    )

    print(
        f"{threshold:>8,} cells"
        f"  →  {contributing_area:.3f} km²"
    )

In [ ]:
# Evaluate the number of drainage cells produced by each threshold

threshold_results = []

for threshold in thresholds:

    stream_mask = flow_acc_valid >= threshold

    stream_cell_count = np.sum(stream_mask)

    stream_area_km2 = (
        stream_cell_count * cell_area_km2
    )

    threshold_results.append({
        "threshold_cells": threshold,
        "contributing_area_km2": threshold * cell_area_km2,
        "stream_cells": int(stream_cell_count),
        "stream_area_km2": stream_area_km2
    })

threshold_results

In [ ]:
import pandas as pd

threshold_df = pd.DataFrame(threshold_results)

threshold_df

## 15. Candidate Drainage Network Comparison

The candidate thresholds are visualized to evaluate how the extracted
drainage network changes as the contributing-area threshold increases.

Lower thresholds should produce denser drainage networks, while higher
thresholds should retain progressively fewer and larger drainage pathways.

The purpose of this comparison is not to select the most visually dense
network, but to identify a threshold that provides a reasonable
representation of the drainage structure for the study area.

In [ ]:
# Visual comparison of candidate drainage networks

fig, axes = plt.subplots(
    4,
    2,
    figsize=(14, 20)
)

axes = axes.flatten()

for i, threshold in enumerate(thresholds):

    stream_mask = flow_acc_valid >= threshold

    axes[i].imshow(
        stream_mask,
        cmap="gray"
    )

    area = threshold * cell_area_km2

    axes[i].set_title(
        f"Threshold = {threshold:,} cells "
        f"({area:.2f} km²)"
    )

    axes[i].set_xlabel("Column")
    axes[i].set_ylabel("Row")

# Hide unused subplot
if len(thresholds) < len(axes):
    for j in range(len(thresholds), len(axes)):
        axes[j].axis("off")

plt.tight_layout()
plt.show()

## 16. Candidate Drainage Network and Gomti River Comparison

The candidate drainage networks are compared with the verified Gomti River
network derived in Notebook 01.

This comparison is a qualitative spatial sanity check rather than a formal
accuracy assessment.

The purpose is to determine whether the selected flow-accumulation
threshold produces a drainage structure that is spatially coherent with the
known Gomti river network.

The Gomti River is not treated as ground truth for every DEM-derived
drainage channel.

In [ ]:
# Candidate thresholds for visual comparison with Gomti
from rasterio.plot import plotting_extent
comparison_thresholds = [
    5_000,
    10_000,
    50_000
]

fig, axes = plt.subplots(
    1,
    3,
    figsize=(20, 7)
)

for ax, threshold in zip(axes, comparison_thresholds):

    stream_mask = flow_acc_valid >= threshold

    ax.imshow(
        stream_mask,
        extent=plotting_extent(
    flow_acc,
    transform=flow_acc_transform
    )
    )

    # Plot Gomti River
    river_distance.plot(
        ax=ax,
        color="red",
        linewidth=0.7
    )

    area = threshold * cell_area_km2

    ax.set_title(
        f"Threshold = {threshold:,} cells\n"
        f"Contributing area = {area:.2f} km²"
    )

    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")

plt.tight_layout()
plt.show()

In [ ]:
threshold_comparison_figure = (
    FIGURES_DIR
    / "drainage_threshold_comparison.png"
)

fig.savefig(
    threshold_comparison_figure,
    dpi=300,
    bbox_inches="tight"
)

print("Threshold comparison figure saved:")
print(threshold_comparison_figure)

## 17. Selected Drainage-Network Threshold

Based on the candidate-threshold analysis, a flow-accumulation threshold
of 10,000 cells is selected for drainage-network extraction.

At the DEM resolution of approximately 28.6 m, 10,000 accumulated cells
represent approximately 8.20 km² of upstream contributing area.

The lower thresholds tested (100–5,000 cells) produced substantially
denser networks containing numerous fine-scale drainage pathways, whereas
the higher thresholds (50,000–100,000 cells) produced overly sparse
networks and removed much of the intermediate drainage structure.

The 10,000-cell threshold provides a balanced representation of the
drainage hierarchy and shows spatial coherence with the verified Gomti
River network.

This threshold is a methodological parameter of the present study and is
not interpreted as a universal stream-initiation threshold.

In [ ]:
# Selected flow-accumulation threshold

SELECTED_THRESHOLD = 10_000

selected_contributing_area_km2 = (
    SELECTED_THRESHOLD * cell_area_km2
)

print("Selected threshold:", SELECTED_THRESHOLD, "cells")
print(
    "Equivalent contributing area:",
    f"{selected_contributing_area_km2:.2f} km²"
)

## 18. Extract the Selected Drainage Network

The selected flow-accumulation threshold is applied to the existing
flow-accumulation raster to create a binary drainage-network raster.

Cells with flow accumulation greater than or equal to 10,000 cells are
classified as drainage cells.

The resulting raster represents DEM-derived drainage pathways within the
analysis domain.

This raster will be used as the basis for subsequent drainage-network
length and drainage-density calculations.

In [ ]:
# Create the selected drainage-network raster

drainage_network = np.zeros(
    flow_acc.shape,
    dtype=np.uint8
)

# Apply threshold only to valid flow-accumulation cells
valid_mask = ~np.ma.getmaskarray(flow_acc_valid)

drainage_network[
    valid_mask & (flow_acc >= SELECTED_THRESHOLD)
] = 1

print("Unique drainage-network values:")
print(np.unique(drainage_network))

print("\nDrainage cells:")
print(np.sum(drainage_network == 1))

print("\nNon-drainage cells:")
print(np.sum(drainage_network == 0))

## 19. Selected Drainage Network Validation

The selected drainage network is visualized together with the verified
Gomti River network.

The purpose is to confirm that the selected network forms a coherent
drainage structure and that the Gomti occupies a major position within the
DEM-derived drainage system.

This is a qualitative validation and does not constitute a formal accuracy
assessment of the extracted drainage network.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 9))

rasterio.plot.show(
    drainage_network,
    ax=ax,
    transform=flow_acc_transform,
    cmap="gray"
)

river_distance.plot(
    ax=ax,
    color="red",
    linewidth=1
)

ax.set_title(
    "Selected Drainage Network "
    "(Flow Accumulation ≥ 10,000 Cells)"
)

ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")

plt.tight_layout()
plt.show()

## 20. Save Selected Drainage Network Diagnostic

The selected drainage network and the verified Gomti River network are
saved as a diagnostic figure.

This figure documents the final drainage-network threshold selected for
the study and provides a visual record of the spatial relationship between
the DEM-derived drainage network and the Gomti River.

The figure is retained as a methodological diagnostic and is not itself
used as a model input.

In [ ]:
# Save the selected drainage network diagnostic figure

drainage_network_figure = (
    FIGURES_DIR
    / "selected_drainage_network_10000_gomti.png"
)

fig.savefig(
    drainage_network_figure,
    dpi=300,
    bbox_inches="tight"
)

print("Selected drainage-network figure saved:")
print(drainage_network_figure)

## 21. Drainage Density — Concept

Drainage density describes the amount of drainage-channel length present
within a given area.

It is defined as:

\[
D_d = \frac{L}{A}
\]

where:

- \(D_d\) = drainage density
- \(L\) = total drainage-channel length
- \(A\) = area considered

Drainage density is commonly expressed in km/km².

A single drainage-density value for the entire study area would not provide
a spatial factor for flood-susceptibility modelling. Therefore, a local
drainage-density raster will be generated by evaluating drainage-channel
length within a moving spatial window.

Higher local drainage density indicates a greater concentration of
drainage channels within the corresponding neighbourhood.

## 22. Drainage Network Centerline Extraction

The selected drainage network is represented as a binary raster in which
drainage cells have a value of 1.

For drainage-density calculation, the drainage network must be represented
as a one-cell-wide centerline rather than as an area of connected raster
cells.

Skeletonization is therefore applied to the binary drainage network.

Skeletonization reduces connected drainage regions to their approximate
centerlines while preserving their overall connectivity and branching
structure.

The resulting skeleton is used to estimate drainage-channel length without
counting the width of the rasterized drainage cells as additional channel
length.

In [ ]:
from skimage.morphology import skeletonize

# Convert selected drainage network to Boolean
drainage_binary = drainage_network == 1

# Skeletonize the drainage network
drainage_skeleton = skeletonize(drainage_binary)

print("Original drainage cells:", np.sum(drainage_binary))
print("Skeleton drainage cells:", np.sum(drainage_skeleton))
print("Skeleton shape:", drainage_skeleton.shape)

## 23. Drainage Skeleton Validation

The skeletonized drainage network is visually inspected to verify that the
one-cell-wide representation preserves the main branching structure of the
selected drainage network.

The skeleton should remain spatially connected to the major drainage
pathways identified using the 10,000-cell flow-accumulation threshold.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 9))

rasterio.plot.show(
    drainage_skeleton.astype(np.uint8),
    ax=ax,
    transform=flow_acc_transform,
    cmap="gray"
)

river_distance.plot(
    ax=ax,
    color="red",
    linewidth=1
)

ax.set_title(
    "Skeletonized Drainage Network "
    "(Flow Accumulation ≥ 10,000 Cells)"
)

ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")

plt.tight_layout()
plt.show()

In [ ]:
skeleton_figure = (
    FIGURES_DIR
    / "drainage_network_skeleton_10000.png"
)

fig.savefig(
    skeleton_figure,
    dpi=300,
    bbox_inches="tight"
)

print("Skeleton diagnostic saved:")
print(skeleton_figure)

## 24. Drainage Channel Length

The skeletonized drainage network is used to estimate drainage-channel
length.

Horizontal and vertical connections between adjacent skeleton cells are
assigned the corresponding raster cell dimension.

Diagonal connections are assigned the diagonal cell distance:

\[
d = \sqrt{(\Delta x)^2 + (\Delta y)^2}
\]

This provides a raster-based estimate of the total drainage-channel length
while accounting for the approximately square 28.6 m analysis cells.

Each connection between adjacent skeleton cells is counted only once.

In [ ]:
pixel_x = abs(flow_acc_transform.a)
pixel_y = abs(flow_acc_transform.e)

diagonal_length = np.sqrt(
    pixel_x**2 + pixel_y**2
)

print(f"Pixel width: {pixel_x:.3f} m")
print(f"Pixel height: {pixel_y:.3f} m")
print(f"Diagonal distance: {diagonal_length:.3f} m")

In [ ]:
# Identify skeleton cells
skeleton = drainage_skeleton

# Horizontal connections
horizontal_connections = (
    skeleton[:, :-1] &
    skeleton[:, 1:]
)

# Vertical connections
vertical_connections = (
    skeleton[:-1, :] &
    skeleton[1:, :]
)

# Diagonal connections
diagonal_down_right = (
    skeleton[:-1, :-1] &
    skeleton[1:, 1:]
)

diagonal_down_left = (
    skeleton[:-1, 1:] &
    skeleton[1:, :-1]
)

# Count connections
horizontal_count = np.sum(horizontal_connections)
vertical_count = np.sum(vertical_connections)
diagonal_right_count = np.sum(diagonal_down_right)
diagonal_left_count = np.sum(diagonal_down_left)

# Calculate total length
horizontal_length = horizontal_count * pixel_x
vertical_length = vertical_count * pixel_y

diagonal_length_total = (
    diagonal_right_count + diagonal_left_count
) * diagonal_length

total_length_m = (
    horizontal_length
    + vertical_length
    + diagonal_length_total
)

total_length_km = total_length_m / 1000

print("Horizontal connections:", horizontal_count)
print("Vertical connections:", vertical_count)
print("Diagonal connections:",
      diagonal_right_count + diagonal_left_count)

print("\nTotal drainage-network length:")
print(f"{total_length_km:.2f} km")

## 25. Local Drainage-Density Window

Drainage density is defined as the total drainage-channel length divided by
the area over which that length is measured:

\[
D_d = \frac{L}{A}
\]

For flood-susceptibility modelling, a single drainage-density value for the
entire study area is insufficient because it does not represent spatial
variation.

A moving spatial window is therefore used to calculate local drainage
density.

For each location:

1. Drainage-channel length within the window is calculated.
2. The drainage length is divided by the window area.
3. The resulting value is assigned to the corresponding raster cell.

The window size controls the spatial scale of the drainage-density factor.

A small window produces highly localized variation, while a large window
produces a smoother regional pattern.

Candidate window sizes will therefore be evaluated before selecting the
final window.

## 26. Candidate Local Drainage-Density Windows

Local drainage density is calculated within a moving square spatial window.

Three candidate window sizes are evaluated:

- 1 km × 1 km
- 2 km × 2 km
- 5 km × 5 km

The corresponding raster window sizes are determined from the analysis
resolution of approximately 28.6 m.

The candidate windows represent different spatial scales of drainage
organization. Smaller windows preserve localized drainage variation,
whereas larger windows produce a smoother regional drainage-density
surface.

The final window size will be selected after visual and statistical
comparison of the resulting drainage-density surfaces.

In [ ]:
# Candidate drainage-density window sizes in metres
window_sizes_m = [1000, 2000, 5000]

print("Analysis resolution:", pixel_x, "m\n")

for window_m in window_sizes_m:

    window_cells = int(round(window_m / pixel_x))

    actual_window_m = window_cells * pixel_x

    print(
        f"Requested: {window_m / 1000:.1f} km"
        f" → {window_cells} × {window_cells} cells"
        f" → actual width: {actual_window_m / 1000:.3f} km"
    )

## 27. Drainage-Length Contribution Raster

Each connection in the skeletonized drainage network contributes its
geometric length to the two cells forming that connection.

To avoid double-counting, half of each connection length is assigned to
each endpoint cell.

Therefore, when the resulting raster is summed over a spatial window, the
total represents the estimated drainage-channel length contained within
that window.

Horizontal and vertical connections use the corresponding raster
dimension, while diagonal connections use the diagonal cell distance.

In [ ]:
# Initialize drainage-length contribution raster
drainage_length_raster = np.zeros(
    skeleton.shape,
    dtype=np.float32
)

# --------------------------------------------------
# Horizontal connections
# --------------------------------------------------

horizontal_connections = (
    skeleton[:, :-1] &
    skeleton[:, 1:]
)

half_horizontal = pixel_x / 2

drainage_length_raster[:, :-1] += (
    horizontal_connections * half_horizontal
)

drainage_length_raster[:, 1:] += (
    horizontal_connections * half_horizontal
)


# --------------------------------------------------
# Vertical connections
# --------------------------------------------------

vertical_connections = (
    skeleton[:-1, :] &
    skeleton[1:, :]
)

half_vertical = pixel_y / 2

drainage_length_raster[:-1, :] += (
    vertical_connections * half_vertical
)

drainage_length_raster[1:, :] += (
    vertical_connections * half_vertical
)


# --------------------------------------------------
# Bottom-right diagonal connections
# --------------------------------------------------

diagonal_down_right = (
    skeleton[:-1, :-1] &
    skeleton[1:, 1:]
)

half_diagonal = diagonal_length / 2

drainage_length_raster[:-1, :-1] += (
    diagonal_down_right * half_diagonal
)

drainage_length_raster[1:, 1:] += (
    diagonal_down_right * half_diagonal
)


# --------------------------------------------------
# Bottom-left diagonal connections
# --------------------------------------------------

diagonal_down_left = (
    skeleton[:-1, 1:] &
    skeleton[1:, :-1]
)

drainage_length_raster[:-1, 1:] += (
    diagonal_down_left * half_diagonal
)

drainage_length_raster[1:, :-1] += (
    diagonal_down_left * half_diagonal
)


print("Drainage-length raster created.")

print(
    "Total length represented:",
    drainage_length_raster.sum() / 1000,
    "km"
)

## 28. Drainage-Length Contribution Validation

The drainage-length contribution raster is checked by comparing its total
length with the previously calculated total skeleton-network length.

Because each drainage connection contributes half its length to each
endpoint cell, summing the entire contribution raster should reproduce the
total drainage-network length without double-counting.

The spatial distribution of the contribution raster is also inspected to
ensure that non-drainage areas contain zero contribution.

In [ ]:
print(
    f"Previous total length: "
    f"{total_length_km:.3f} km"
)

contribution_length_km = (
    drainage_length_raster.sum() / 1000
)

print(
    f"Contribution raster total: "
    f"{contribution_length_km:.3f} km"
)

difference_km = (
    contribution_length_km - total_length_km
)

print(
    f"Difference: {difference_km:.6f} km"
)

## 29. Local Drainage-Density Calculation

Local drainage density is calculated by summing the drainage-channel length
within a moving spatial window and dividing it by the area of that window.

\[
D_d = \frac{L_w}{A_w}
\]

where:

- \(L_w\) = drainage-channel length within the moving window
- \(A_w\) = area of the moving window

Three candidate spatial scales are evaluated:

- 1 km × 1 km
- 2 km × 2 km
- 5 km × 5 km

The resulting drainage-density surfaces will be compared to determine
whether the spatial pattern is stable and meaningful for flood-susceptibility
modelling.

In [ ]:
from scipy.ndimage import uniform_filter

In [ ]:
# Candidate window sizes
window_sizes_km = [1, 2, 5]

window_info = []

for window_km in window_sizes_km:

    window_cells = int(
        round((window_km * 1000) / pixel_x)
    )

    # Ensure an odd number of cells so that the focal cell
    # remains at the centre of the moving window
    if window_cells % 2 == 0:
        window_cells += 1

    actual_window_m = window_cells * pixel_x
    actual_window_km = actual_window_m / 1000

    window_info.append({
        "requested_window_km": window_km,
        "window_cells": window_cells,
        "actual_window_km": actual_window_km
    })

window_info

In [ ]:
# Dictionary to store drainage-density rasters
drainage_density_candidates = {}

for info in window_info:

    window_km = info["requested_window_km"]
    window_cells = info["window_cells"]

    # Mean drainage length contribution per cell
    local_mean_length_m = uniform_filter(
        drainage_length_raster,
        size=window_cells,
        mode="constant",
        cval=0.0
    )

    # Convert mean back to total drainage length within the window
    local_length_m = (
        local_mean_length_m
        * window_cells
        * window_cells
    )

    # Window area in km²
    actual_window_area_km2 = (
        (window_cells * pixel_x)
        * (window_cells * pixel_y)
        / 1_000_000
    )

    # Convert local drainage length to km
    local_length_km = local_length_m / 1000

    # Drainage density = length / area
    drainage_density = (
        local_length_km
        / actual_window_area_km2
    )

    drainage_density_candidates[window_km] = (
        drainage_density.astype(np.float32)
    )

    print(
        f"{window_km} km window "
        f"({window_cells} × {window_cells} cells)"
    )

    print(
        f"  Window area: "
        f"{actual_window_area_km2:.3f} km²"
    )

    print(
        f"  Min density: "
        f"{drainage_density.min():.4f} km/km²"
    )

    print(
        f"  Max density: "
        f"{drainage_density.max():.4f} km/km²"
    )

    print(
        f"  Mean density: "
        f"{drainage_density.mean():.4f} km/km²"
    )

    print()

## 30. Candidate Drainage-Density Surface Comparison

The drainage-density rasters generated using the 1 km, 2 km, and 5 km
moving windows are compared spatially.

Smaller windows are expected to preserve more localized variation in
drainage concentration, while larger windows should produce smoother
regional patterns.

The comparison is used to identify a spatial scale that provides useful
variation without producing excessive local noise.

In [ ]:
fig, axes = plt.subplots(
    1,
    3,
    figsize=(20, 7)
)

for ax, window_km in zip(
    axes,
    window_sizes_km
):

    density = drainage_density_candidates[
        window_km
    ]

    rasterio.plot.show(
        density,
        ax=ax,
        transform=flow_acc_transform,
        cmap="viridis"
    )

    ax.set_title(
        f"Drainage Density — {window_km} km Window"
    )

    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")

plt.tight_layout()
plt.show()

In [ ]:
for window_km in window_sizes_km:

    density = drainage_density_candidates[
        window_km
    ]

    print(f"\n{window_km} km window")

    print(
        f"Minimum: {np.min(density):.4f} km/km²"
    )

    print(
        f"Maximum: {np.max(density):.4f} km/km²"
    )

    print(
        f"Mean: {np.mean(density):.4f} km/km²"
    )

    print(
        f"Median: {np.median(density):.4f} km/km²"
    )

    print(
        f"Std: {np.std(density):.4f} km/km²"
    )

## 31. Drainage-Density Window Selection

The candidate drainage-density surfaces were evaluated using three spatial
window sizes: 1 km, 2 km, and 5 km.

The 1 km window produced highly localized and fragmented drainage-density
patterns, with a median density of 0 km/km².

The 5 km window produced a substantially smoother surface and reduced local
spatial variation.

The 2 km window provided an intermediate spatial scale, retaining meaningful
local drainage variation without the extreme fragmentation observed with
the 1 km window.

Therefore, the 2 km × 2 km window is selected for the final
drainage-density factor.

This selection is specific to the present study and is based on the
sensitivity analysis of the tested spatial scales.

In [ ]:
# Selected drainage-density surface
SELECTED_DD_WINDOW_KM = 2

selected_drainage_density = (
    drainage_density_candidates[
        SELECTED_DD_WINDOW_KM
    ]
)

print(
    "Selected window:",
    SELECTED_DD_WINDOW_KM,
    "km ×",
    SELECTED_DD_WINDOW_KM,
    "km"
)

print(
    "Minimum:",
    selected_drainage_density.min(),
    "km/km²"
)

print(
    "Maximum:",
    selected_drainage_density.max(),
    "km/km²"
)

print(
    "Mean:",
    selected_drainage_density.mean(),
    "km/km²"
)

print(
    "Median:",
    np.median(selected_drainage_density),
    "km/km²"
)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 9))

rasterio.plot.show(
    selected_drainage_density,
    ax=ax,
    transform=flow_acc_transform,
    cmap="viridis"
)

river_distance.plot(
    ax=ax,
    color="red",
    linewidth=1
)

ax.set_title(
    "Selected Drainage Density — 2 km Window\n"
    "with Gomti River Network"
)

ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")

plt.tight_layout()
plt.show()

## 32. Clip Drainage Density to the Study Area

The selected 2 km local drainage-density surface is clipped to the
established Lucknow 20 km study area.

The hydrological calculations were performed on the expanded raster domain
to avoid edge effects during terrain and moving-window processing.

Only the portion corresponding to the defined study area is retained as
the final drainage-density conditioning factor.

Cells outside the study area are assigned NoData.

In [ ]:
# Load the established 20 km study area

study_area = gpd.read_file(
    PROJECT_ROOT/ "data/processed/notebook_01/Lucknow_Search_Area_20km.gpkg"
)

print("Study area CRS:", study_area.crs)
print("Study area features:", len(study_area))

In [ ]:
# Assign the analysis CRS if the study-area file has no CRS metadata

if study_area.crs is None:
    study_area = study_area.set_crs(flow_acc_crs)

# Reproject study area to the drainage-density CRS if necessary
elif study_area.crs != flow_acc_crs:
    study_area = study_area.to_crs(flow_acc_crs)

print("Study area CRS after check:", study_area.crs)
print("Drainage-density CRS:", flow_acc_crs)

## 33. Final Drainage-Density Raster

The selected 2 km drainage-density surface is masked using the established
study-area boundary.

The resulting raster represents local drainage density only within the
defined study area and retains the spatial resolution, CRS, and grid
alignment of the hydrological analysis raster.

In [ ]:
from rasterio.features import geometry_mask

# Create mask for the study area

study_area_mask = geometry_mask(
    study_area.geometry,
    out_shape=selected_drainage_density.shape,
    transform=flow_acc_transform,
    invert=True
)

# Apply study-area mask
final_drainage_density = np.where(
    study_area_mask,
    selected_drainage_density,
    np.nan
).astype(np.float32)

print("Final drainage-density raster created.")

print(
    "Valid cells:",
    np.sum(~np.isnan(final_drainage_density))
)

print(
    "NoData cells:",
    np.sum(np.isnan(final_drainage_density))
)

## 34. Final Drainage-Density Validation

The clipped drainage-density raster is checked to ensure that valid values
occur only within the defined study area.

The raster statistics are recalculated after masking because the final
conditioning factor should describe the study area rather than the expanded
hydrological processing domain.

The raster CRS, dimensions, resolution, transform, and spatial extent are
also checked against the established analysis grid.

In [ ]:
valid_density = final_drainage_density[
    ~np.isnan(final_drainage_density)
]

print("Final drainage-density statistics")
print("-----------------------------------")

print(
    f"Minimum: {valid_density.min():.4f} km/km²"
)

print(
    f"Maximum: {valid_density.max():.4f} km/km²"
)

print(
    f"Mean: {valid_density.mean():.4f} km/km²"
)

print(
    f"Median: {np.median(valid_density):.4f} km/km²"
)

print(
    f"Std: {valid_density.std():.4f} km/km²"
)

print("\nRaster geometry")
print("----------------")

print("CRS:", flow_acc_crs)
print("Width:", flow_acc.shape[1])
print("Height:", flow_acc.shape[0])
print("Resolution:", (pixel_x, pixel_y))

## 35. Save Final Drainage-Density Raster

The validated drainage-density surface is saved as a GeoTIFF for use as a
conditioning factor in the subsequent flood-susceptibility workflow.

The raster retains the hydrological analysis CRS, spatial resolution, grid
alignment, and study-area extent.

In [ ]:
# Output path
drainage_density_output = (
    PROJECT_ROOT/ "data" / "processed" / "notebook_03"
    / "Drainage_Density_2km.tif"
)

# Start from the flow-accumulation raster profile
drainage_density_profile = flow_acc_profile.copy()

drainage_density_profile.update(
    dtype="float32",
    count=1,
    nodata=-9999.0,
    compress="lzw"
)

# Replace NaN with NoData value
drainage_density_to_save = np.where(
    np.isnan(final_drainage_density),
    -9999.0,
    final_drainage_density
).astype(np.float32)

with rasterio.open(
    drainage_density_output,
    "w",
    **drainage_density_profile
) as dst:

    dst.write(
        drainage_density_to_save,
        1
    )

print("Final drainage-density raster saved:")
print(drainage_density_output)

## 36. Final Drainage-Density Diagnostic

The final study-area drainage-density raster is visualized together with
the Gomti River network.

This figure documents the spatial distribution of the selected drainage
density factor and its relationship with the verified Gomti River network.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 9))

rasterio.plot.show(
    final_drainage_density,
    ax=ax,
    transform=flow_acc_transform,
    cmap="viridis"
)

river_distance.plot(
    ax=ax,
    color="red",
    linewidth=1
)

ax.set_title(
    "Final Drainage Density — 2 km Window\n"
    "Gomti River Network"
)

ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")

plt.tight_layout()
plt.savefig(FIGURES_DIR/"final_drainage_density_2km_gomti.png")
plt.show()

# Notebook 03 — Rivers & Hydrological Factors
## Final Summary

### 1. Objective

Notebook 03 was developed to generate and validate the river-related and
drainage-related hydrological factors required for the flood-susceptibility
analysis of the Gomti River-influenced study area around Lucknow.

The notebook builds upon:

- the verified Gomti River network from Notebook 01
- the DEM-derived hydrological products from Notebook 02

The objective was to generate only those hydrological factors that have a
clear methodological role in the subsequent flood-susceptibility model.

---

## 2. Verified Gomti River Network

The Gomti River network established in Notebook 01 was used as the reference
river network.

The selected network contains:

- 160 Gomti river reaches
- approximately 715.42 km of total river length

The complete selected Gomti network falls within the defined
Lucknow + 20 km study area.

The river network was retained as the reference layer for river-distance
analysis and spatial comparison with the DEM-derived drainage network.

---

## 3. DEM-Derived Drainage Network

The conditioned DEM and flow-accumulation raster generated in Notebook 02
were used to derive a drainage network.

Several flow-accumulation thresholds were investigated to understand how the
derived drainage structure changes with contributing area.

The tested thresholds were:

| Threshold | Contributing Area |
|-----------|-------------------|
| 100 cells | 0.08 km² |
| 500 cells | 0.41 km² |
| 1,000 cells | 0.82 km² |
| 5,000 cells | 4.10 km² |
| 10,000 cells | 8.20 km² |
| 50,000 cells | 41.00 km² |
| 100,000 cells | 82.00 km² |

The threshold analysis showed that lower thresholds produce very dense
drainage structures, while higher thresholds progressively retain only the
larger drainage channels.

A threshold of 10,000 flow-accumulation cells was selected for the final
drainage network used in the subsequent analysis.

This corresponds to a contributing area of approximately:

**8.20 km²**

The selected drainage network was spatially examined together with the
Gomti River network to verify that the derived drainage structure was
hydrologically plausible.

---

## 4. Drainage Network Length

The selected drainage network was skeletonized to obtain a one-cell-wide
representation of the DEM-derived drainage pathways.

Drainage-channel length was estimated from the connectivity of the
skeletonized raster by accounting for horizontal, vertical, and diagonal
cell-to-cell connections.

The calculated total drainage-network length was approximately:

**4,635.74 km**

A drainage-length contribution raster was then generated by assigning half
of each connection length to each of its two endpoint cells.

The total length represented by this contribution raster was:

- Skeleton connectivity estimate: approximately 4,635.738 km
- Drainage-length contribution raster: approximately 4,635.737 km
- Difference: approximately -0.0004 km

The negligible difference confirms that the drainage-length contribution
raster preserves the same total network length without double-counting the
cell-to-cell connections.

---

## 5. Drainage Density

Drainage density was calculated as:

\[
D_d = \frac{L}{A}
\]

where:

- \(D_d\) = drainage density
- \(L\) = drainage-network length within the analysis window
- \(A\) = area of the analysis window

Drainage density was evaluated using multiple spatial window sizes:

- 1 km × 1 km
- 2 km × 2 km
- 5 km × 5 km

The results were:

| Window | Minimum (km/km²) | Maximum (km/km²) | Mean (km/km²) | Median (km/km²) | Std (km/km²) |
|--------|------------------:|------------------:|---------------:|-----------------:|-------------:|
| 1 km | 0.0000 | 4.1100 | 0.3588 | 0.0000 | 0.5849 |
| 2 km | 0.0000 | 2.3820 | 0.3581 | 0.2454 | 0.3823 |
| 5 km | 0.0000 | 1.0659 | 0.3556 | 0.3522 | 0.1904 |

The 1 km window provides a highly localized drainage-density pattern but
contains many zero-density cells and greater local variability.

The 5 km window produces a substantially smoother spatial pattern.

The 2 km window provides a compromise between local spatial detail and
smoothing and was therefore selected as the drainage-density representation
for the subsequent workflow.

Selected drainage-density window:

**2 km × 2 km**

Final statistics:

- Minimum: 0.0000 km/km²
- Maximum: 2.3820 km/km²
- Mean: 0.3581 km/km²
- Median: 0.2454 km/km²

---

## 6. River and Drainage Relationship

The final drainage-density raster was visualized together with the verified
Gomti River network.

This comparison was used as a spatial sanity check rather than as a formal
accuracy assessment.

The DEM-derived drainage network and the Gomti River network show a
spatially plausible hydrological structure.

The drainage network is therefore retained as a supporting hydrological
factor rather than being treated as an exact reconstruction of the observed
river network.

---

## 7. Hydrological Factors Produced

Notebook 03 produces the following important hydrological information:

### Existing input from Notebook 02

- Flow accumulation

### Generated in Notebook 03

- Selected DEM-derived drainage network
- Drainage-length representation
- 2 km drainage-density raster
- Verified Gomti River network used for river-related analysis

The selected Gomti River network remains the reference river layer, while
the DEM-derived drainage network represents the spatial pattern of
topographically derived runoff convergence.

---

## 8. Methodological Decision

The notebook does **not** introduce additional hydrological variables simply
because they can be calculated.

The selected hydrological information is limited to factors that have a
clear role in flood susceptibility:

1. Flow accumulation
2. Drainage density
3. Distance/proximity to the Gomti River or drainage network

This keeps the workflow focused and avoids unnecessary predictor variables.

---

## 9. Validation and Quality Checks

The following checks were performed during Notebook 03:

- drainage-network spatial inspection
- flow-accumulation threshold comparison
- comparison of drainage network with the Gomti River network
- drainage-network length calculation
- skeleton-length and drainage-length-raster consistency check
- comparison of drainage-density window sizes
- visual inspection of the selected 2 km drainage-density raster

The drainage-length consistency check showed a negligible difference between
the skeleton-derived and drainage-length-raster total length

The hydrological products are therefore considered suitable for progression
to the next stage of the project.

---

# 10. Final Status

Notebook 03 — Rivers & Hydrological Factors

**STATUS: COMPLETE**

The outputs from Notebook 03 will be used together with the terrain and
hydrological products from Notebook 02 in the later flood-susceptibility
workflow.

The next stage is:

**Notebook 04 — Land Use / Land Cover**

Notebook 04 will introduce the temporal component of the study by preparing
comparable LULC datasets for:

- 2003
- 2014
- 2025

The LULC datasets must use a consistent classification framework so that
urbanization and land-cover changes can be compared meaningfully across
the three temporal stages.